# ComfyUI + IPAdapter + WAI-Illustrious-SDXL (Colab)

Hücre sırası:
1. **CONFIG** — token'lar ve workspace ayarı (her yeni oturumda buraya tokenları gir)
2. Ortam kurulumu (ComfyUI + Manager)
3. IPAdapter custom node
4. IPAdapter + CLIP Vision indir (HuggingFace, sorunsuz)
5. **Checkpoint indir** — izole, curl ile (Civitai 400 hatası buraya özel çözüldü)
6. Tüm dosyaları kontrol et
7. ComfyUI'yi başlat

## 1) CONFIG — token'lar ve workspace

**Bu hücreyi her oturumda en az bir kez çalıştır.** Token girip çalıştırınca os.environ'a yazar, sonraki hücreler buradan okur.

- **CIVITAI_TOKEN:** https://civitai.com → Account Settings → API Keys → Add API Key
- **HF_TOKEN:** (opsiyonel) sadece gated HuggingFace modelleri için gerekir, IPAdapter / CLIP Vision için gerekmez

In [ ]:
# @title 🔑 Tokens & paths
import os

CIVITAI_TOKEN = "c2f8440d15d825229098e9a5a75c5dfa"  #@param {type:"string"}
HF_TOKEN      = ""  #@param {type:"string"}

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}

os.environ['CIVITAI_TOKEN'] = CIVITAI_TOKEN
os.environ['HF_TOKEN']      = HF_TOKEN

if USE_GOOGLE_DRIVE:
    WORKSPACE = '/content/drive/MyDrive/ComfyUI'
else:
    WORKSPACE = '/content/ComfyUI'
os.environ['WORKSPACE'] = WORKSPACE

print(f"WORKSPACE     = {WORKSPACE}")
print(f"CIVITAI_TOKEN = {'(set, ' + str(len(CIVITAI_TOKEN)) + ' chars)' if CIVITAI_TOKEN else '(BOŞ — checkpoint inmez!)'}")
print(f"HF_TOKEN      = {'(set)' if HF_TOKEN else '(boş — sorun değil)'}")

## 2) Ortam kurulumu (ComfyUI + Manager)

In [ ]:
import os
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')

if WORKSPACE.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive
else:
    %cd /content

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

!echo -= Updating ComfyUI =-
!git pull

# ComfyUI'nin kendi requirements'ı (comfy_aimdo vs için kritik)
!echo -= Install ComfyUI requirements =-
!pip install -r requirements.txt
!pip install blake3

# ComfyUI-Manager
%cd custom_nodes
![ ! -d ComfyUI-Manager ] && git clone https://github.com/ltdrdata/ComfyUI-Manager
%cd ComfyUI-Manager && !git pull

%cd $WORKSPACE
!pip install GitPython
!python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies

## 3) IPAdapter custom node

In [ ]:
import os
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')
%cd $WORKSPACE/custom_nodes

![ ! -d ComfyUI_IPAdapter_plus ] && git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus
%cd ComfyUI_IPAdapter_plus
!git pull
![ -f requirements.txt ] && pip install -r requirements.txt

%cd $WORKSPACE

## 4) IPAdapter + CLIP Vision indir (HuggingFace)
Bunlar sorunsuz iniyor, login gerekmiyor.

In [ ]:
import os
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')
%cd $WORKSPACE

!mkdir -p models/ipadapter models/clip_vision models/checkpoints/sdxl

# IPAdapter (SDXL)
!wget -c https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors \
  -O models/ipadapter/ip-adapter-plus_sdxl_vit-h.safetensors

# CLIP Vision (ViT-H)
!wget -c https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors \
  -O models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors

!echo "--- IPAdapter ---" && ls -lh models/ipadapter/
!echo "--- CLIP Vision ---" && ls -lh models/clip_vision/

## 5) Checkpoint indir — IZOLE, curl ile

Bu hücre **sadece WAI-Illustrious checkpoint'ini** indirir.
Hata alırsan tekrar tekrar çalıştırırken bütün model setini yeniden indirmen gerekmez.

**Strateji:**
- `curl -L` → redirect (Cloudflare R2) takibi
- Token hem `Authorization: Bearer` header'ı hem URL parametresi olarak (Civitai bazen birini bazen diğerini istiyor)
- Tarayıcı User-Agent → bazı CDN'ler boş UA'ya 400 dönüyor
- `-C -` → indirme yarıda kesilirse devam eder
- `--fail` → HTTP hatasını sessizce yutmaz, kod 0'dan farklı döner

**Token girmedi̇ysen önce CONFIG hücresini düzelt!**

In [ ]:
import os, sys
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')
TOKEN = os.environ.get('CIVITAI_TOKEN', '')
%cd $WORKSPACE

if not TOKEN:
    print("⚠️  CIVITAI_TOKEN boş — CONFIG hücresine token girip o hücreyi çalıştır, sonra buraya dön.")
else:
    !mkdir -p models/checkpoints/sdxl
    # Bozuk dosyayı temizle (varsa)
    !rm -f models/checkpoints/sdxl/waiIllustrious.safetensors

    MODEL_VERSION_ID = 2883731  # WAI-Illustrious-SDXL
    URL = f"https://civitai.com/api/download/models/{MODEL_VERSION_ID}?token={TOKEN}"
    OUT = "models/checkpoints/sdxl/waiIllustrious.safetensors"

    os.environ['DL_URL'] = URL
    os.environ['DL_OUT'] = OUT

    !curl -L --fail -C - \
        -H "Authorization: Bearer $CIVITAI_TOKEN" \
        -H "User-Agent: Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36" \
        -o "$DL_OUT" \
        "$DL_URL"

    # Hızlı boyut kontrolü
    if os.path.exists(OUT):
        size_gb = os.path.getsize(OUT) / 1024 / 1024 / 1024
        if size_gb < 1:
            print(f"\n⚠️  Dosya çok küçük ({size_gb*1024:.1f} MB). Token doğru mu kontrol et.")
        else:
            print(f"\n✓ İndirme başarılı: {size_gb:.2f} GB")
    else:
        print("\n✗ Dosya hiç oluşmadı. Curl çıktısına bak.")

## 6) Tüm dosyaları topluca kontrol et

In [ ]:
import os
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')

checks = [
    (f'{WORKSPACE}/models/checkpoints/sdxl/waiIllustrious.safetensors', 1_000_000_000, 'Checkpoint (~6.5 GB)'),
    (f'{WORKSPACE}/models/ipadapter/ip-adapter-plus_sdxl_vit-h.safetensors', 100_000_000, 'IPAdapter (~800 MB)'),
    (f'{WORKSPACE}/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors', 1_000_000_000, 'CLIP Vision (~2.5 GB)'),
]

all_ok = True
print("="*60)
for path, min_size, desc in checks:
    if not os.path.exists(path):
        print(f"❌ YOK: {desc}")
        all_ok = False
    else:
        size_gb = os.path.getsize(path) / 1024 / 1024 / 1024
        ok = os.path.getsize(path) >= min_size
        mark = "✓" if ok else "⚠️"
        print(f"{mark} {desc:35s} {size_gb:.2f} GB")
        if not ok:
            all_ok = False
print("="*60)
print("✓ Hepsi hazır, ComfyUI'yi başlatabilirsin." if all_ok else "✗ Eksik var, ilgili indirme hücresini tekrar çalıştır.")

## 7) ComfyUI'yi başlat (Cloudflared)
Log'da çıkan `trycloudflare.com` linkine tıkla → **Load** → `ipadapter_illustrious_sdxl.json` → görseli yükle → **Queue Prompt**.

In [ ]:
import os
WORKSPACE = os.environ.get('WORKSPACE', '/content/ComfyUI')
%cd $WORKSPACE

!wget -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI yüklendi, cloudflared başlatılıyor...\n")
  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("ComfyUI link:", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server